# 🫁 ICBHI 2017 — Federated Learning v4
## 2D CNN + Two-Stage Cascade + Full Ablation Study

### Key upgrades from v3
- ✅ **2D CNN (RespCNN)** — STFT+LogMel fed as 2-channel 64×64 image, spatial patterns preserved
- ✅ **Dedicated Stage 1 binary CNN** — trained only for Normal vs Abnormal
- ✅ **Auto threshold selection** from validation sweep
- ✅ **EarlyStopping + ReduceLROnPlateau** in centralized and Stage 2 training
- ✅ **Full ablation suite** — threshold, cap_ratio, gamma, LR, all auto-selecting best values
- ✅ **Final retrain** with best ablation parameters
- ✅ **Before vs After comparison** cell

---
Run cells **top to bottom** in order.


## Step 1 — Imports & Environment Check


In [1]:
import subprocess, sys
for pkg in ['PyWavelets', 'scikit-image', 'scipy', 'seaborn']:
    subprocess.run([sys.executable, '-m', 'pip', 'install', pkg, '-q'], check=True)
print('✓ Packages ready')

✓ Packages ready


In [2]:
import sys, os, copy, json, warnings, random, itertools
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset, Dataset
from pathlib import Path
from tqdm import tqdm

import librosa
from scipy.signal import butter, sosfiltfilt
from skimage.transform import resize

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    classification_report, confusion_matrix, f1_score
)

PROJECT_ROOT = Path.home() / 'icbhi_fl'
PROJECT_ROOT.mkdir(exist_ok=True)
os.chdir(PROJECT_ROOT)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f'Working directory : {os.getcwd()}')
print(f'Python            : {sys.version.split()[0]}')
print(f'PyTorch           : {torch.__version__}')
print(f'Librosa           : {librosa.__version__}')
print(f'Device            : {DEVICE}')
print('\n✓ All imports successful')


Working directory : /home/ramima/icbhi_fl
Python            : 3.10.20
PyTorch           : 2.2.2+cu121
Librosa           : 0.11.0
Device            : cpu

✓ All imports successful


## Step 2 — Locate Dataset


In [3]:
DATA_DIR = None
candidates = [
    Path('data/ICBHI_final_database'),
    Path('data/audio_and_txt_files'),
    Path('data/Respiratory_Sound_Database/audio_and_txt_files'),
    Path('data/respiratory-sound-database/audio_and_txt_files'),
    Path('data/ICBHI_final_database/ICBHI_final_database'),
    Path('data'),
]
for c in candidates:
    wavs = list(c.glob('*.wav')) if c.exists() else []
    if len(wavs) > 10:
        DATA_DIR = c
        break
if DATA_DIR is None:
    for p in sorted(Path('data').rglob('*.wav'))[:1]:
        DATA_DIR = p.parent

if DATA_DIR is None:
    print('✗  Cannot find audio files under ~/icbhi_fl/data/')
else:
    wav_files = sorted(DATA_DIR.glob('*.wav'))
    print(f'✓  Data folder : {DATA_DIR}')
    print(f'   WAV files   : {len(wav_files)}')
    print('\n✅ Complete!' if len(wav_files) >= 900
          else f'\n⚠  Expected ~920, found {len(wav_files)}')


✓  Data folder : data/ICBHI_final_database
   WAV files   : 920

✅ Complete!


## Step 3 — Configuration


In [4]:
# ── Paths ─────────────────────────────────────────────────────────────────────
CACHE_FILE  = Path('data/icbhi_cache_v4.npz')   # new cache — 2-channel images
RESULTS_DIR = Path('results')
RESULTS_DIR.mkdir(exist_ok=True)

# ── Audio ─────────────────────────────────────────────────────────────────────
TARGET_SR    = 4000
CLIP_SEC     = 3.0
CLIP_SAMPLES = int(TARGET_SR * CLIP_SEC)

# ── Filter ────────────────────────────────────────────────────────────────────
FILTER_ORDER = 3
LOWCUT       = 100
HIGHCUT      = 1900

# ── Spectrogram ───────────────────────────────────────────────────────────────
N_FFT      = 256
HOP_LENGTH = 64
N_MELS     = 128
FMIN       = 100
FMAX       = 1900
IMAGE_H    = 128     # CNN input height
IMAGE_W    = 128    # CNN input width
N_CHANNELS = 2      # channel 0 = STFT, channel 1 = LogMel

# ── Labels ────────────────────────────────────────────────────────────────────
LABEL_MAP   = {(0,0): 0, (1,0): 1, (0,1): 2, (1,1): 3}
LABEL_NAMES = ['Normal', 'Crackle', 'Wheeze', 'Both']
NUM_CLASSES = 4

# ── FL hyperparameters ────────────────────────────────────────────────────────
MIN_CYCLES   = 5
NUM_ROUNDS   = 20
LOCAL_EPOCHS = 3
LR           = 1e-3
BATCH_SIZE   = 8    # smaller batch for 2D CNN memory
MU_SWEEP     = [0.1, 0.5, 1.0]

# ── Cascade defaults (will be overwritten by ablation) ────────────────────────
STAGE1_THRESHOLD = 0.55

print('✓ Configuration set')
print(f'  Clip          : {CLIP_SEC}s → {CLIP_SAMPLES} samples')
print(f'  Bandpass      : {LOWCUT}–{HIGHCUT} Hz')
print(f'  CNN input     : ({N_CHANNELS}, {IMAGE_H}, {IMAGE_W})')
print(f'  FL rounds     : {NUM_ROUNDS} × {LOCAL_EPOCHS} local epochs')
print(f'  Device        : {DEVICE}')


✓ Configuration set
  Clip          : 3.0s → 12000 samples
  Bandpass      : 100–1900 Hz
  CNN input     : (2, 128, 128)
  FL rounds     : 20 × 3 local epochs
  Device        : cpu


## Step 4 — Preprocessing Functions

Key difference from v3: `extract_image()` returns a **(2, 64, 64) tensor** — STFT as channel 0,  
LogMel as channel 1. The 2D spatial structure is fully preserved for the CNN.


In [5]:
def parse_annotation(txt_path):
    cycles = []
    raw = txt_path.read_text(encoding='utf-8', errors='ignore')
    for line in raw.strip().splitlines():
        line = line.strip()
        if not line:
            continue
        parts = line.split('\t') if '\t' in line else line.split()
        if len(parts) < 4:
            continue
        try:
            cycles.append((
                float(parts[0]), float(parts[1]),
                int(float(parts[2])), int(float(parts[3]))
            ))
        except (ValueError, IndexError):
            continue
    return cycles


def butter_bandpass_filter(audio, lowcut=LOWCUT, highcut=HIGHCUT,
                            fs=TARGET_SR, order=FILTER_ORDER):
    sos = butter(order, [lowcut, highcut], btype='bandpass', fs=fs, output='sos')
    return sosfiltfilt(sos, audio).astype(np.float32)


def normalize_amplitude(signal):
    max_val = np.abs(signal).max()
    return signal / max_val if max_val > 1e-8 else signal


def augment_audio(audio):
    """From teammate Cell 22 — gain + time shift + Gaussian noise."""
    aug = audio.copy()
    aug = aug * np.random.uniform(0.90, 1.10)
    shift = np.random.randint(-int(0.05 * TARGET_SR), int(0.05 * TARGET_SR))
    aug = np.roll(aug, shift)
    aug = aug + np.random.normal(0, 0.005 * np.std(aug), size=aug.shape)
    max_val = np.abs(aug).max()
    if max_val > 1:
        aug = aug / max_val
    return aug.astype(np.float32)


def resize_feature(feature):
    return resize(feature, (IMAGE_H, IMAGE_W), anti_aliasing=True).astype(np.float32)


def extract_image(signal, sr, do_augment=False):
    """
    Returns (2, IMAGE_H, IMAGE_W) float32 array.
    Channel 0 = STFT magnitude in dB
    Channel 1 = Log-Mel spectrogram
    This is the KEY difference from v3 — full 2D image, not flattened stats.
    """
    # Pad / trim
    if len(signal) < CLIP_SAMPLES:
        signal = np.pad(signal, (0, CLIP_SAMPLES - len(signal)))
    else:
        signal = signal[:CLIP_SAMPLES]

    signal = normalize_amplitude(signal)

    if do_augment:
        signal = augment_audio(signal)

    sig = signal.astype(np.float32)

    # Channel 0: STFT
    stft    = librosa.stft(y=sig, n_fft=N_FFT, hop_length=HOP_LENGTH,
                            window='hann', center=True)
    stft_db = librosa.amplitude_to_db(np.abs(stft), ref=np.max)
    ch0     = resize_feature(stft_db)   # (IMAGE_H, IMAGE_W)

    # Channel 1: Log-Mel
    mel     = librosa.feature.melspectrogram(
        y=sig, sr=sr, n_fft=N_FFT, hop_length=HOP_LENGTH,
        n_mels=N_MELS, fmin=FMIN, fmax=FMAX, power=2.0
    )
    log_mel = librosa.power_to_db(mel, ref=np.max)
    ch1     = resize_feature(log_mel)   # (IMAGE_H, IMAGE_W)

    # Stack into (2, IMAGE_H, IMAGE_W)
    return np.stack([ch0, ch1], axis=0)


print('✓ Preprocessing functions ready')
print(f'  extract_image() output shape: ({N_CHANNELS}, {IMAGE_H}, {IMAGE_W})')
print('  Channel 0 = STFT dB   | Channel 1 = Log-Mel')


✓ Preprocessing functions ready
  extract_image() output shape: (2, 128, 128)
  Channel 0 = STFT dB   | Channel 1 = Log-Mel


## Step 5 — Load Data & Extract Features


In [6]:
def load_dataset(data_dir, cache_file):
    if cache_file.exists():
        print('Loading from cache ...')
        d = np.load(cache_file, allow_pickle=True)
        return d['X'], d['y'], d['patients'].tolist()

    print('Processing audio (first run — 15–25 min) ...')
    wav_files = sorted(data_dir.glob('*.wav'))
    print(f'  Found {len(wav_files)} WAV files')

    X_list, y_list, p_list = [], [], []
    skipped_no_txt = skipped_no_cycles = errors = 0

    for wav_path in tqdm(wav_files, desc='Extracting images'):
        txt_path = wav_path.with_suffix('.txt')
        if not txt_path.exists():
            skipped_no_txt += 1
            continue

        patient_id = wav_path.stem.split('_')[0]
        cycles = parse_annotation(txt_path)
        if not cycles:
            skipped_no_cycles += 1
            continue

        try:
            signal, sr = librosa.load(wav_path, sr=None, mono=True)
        except Exception:
            errors += 1
            continue

        for (start, end, crackle, wheeze) in cycles:
            s = int(start * sr)
            e = min(int(end * sr), len(signal))
            if e <= s:
                continue
            cycle_sig = signal[s:e]
            cycle_sig = librosa.resample(cycle_sig, orig_sr=sr, target_sr=TARGET_SR)
            cycle_sig = butter_bandpass_filter(cycle_sig)
            try:
                img   = extract_image(cycle_sig, TARGET_SR, do_augment=False)
                label = LABEL_MAP.get((crackle, wheeze), 0)
                X_list.append(img)
                y_list.append(label)
                p_list.append(patient_id)
            except Exception:
                continue

    print(f'  Skipped (no .txt)   : {skipped_no_txt}')
    print(f'  Skipped (no cycles) : {skipped_no_cycles}')
    print(f'  Errors              : {errors}')

    if len(X_list) == 0:
        print('❌ Zero cycles extracted.')
        return np.array([]), np.array([]), []

    X = np.array(X_list, dtype=np.float32)   # (N, 2, 64, 64)
    y = np.array(y_list, dtype=np.int64)
    np.savez(cache_file, X=X, y=y, patients=np.array(p_list))
    print(f'✓ Saved cache: {cache_file}')
    return X, y, p_list


assert DATA_DIR is not None, 'DATA_DIR not set — re-run Step 2'
X_all, y_all, patients_all = load_dataset(DATA_DIR, CACHE_FILE)

if len(X_all) == 0:
    raise RuntimeError('Zero cycles loaded.')

print(f'\n✅ Dataset loaded')
print(f'   Total cycles   : {len(X_all)}')
print(f'   Image shape    : {X_all.shape[1:]}  (channels, H, W)')
print(f'   Unique patients: {len(set(patients_all))}')
unique, counts = np.unique(y_all, return_counts=True)
print('\nClass distribution:')
for u, c in zip(unique, counts):
    print(f'   {LABEL_NAMES[u]:10s}: {c:5d}  ({100*c/len(y_all):.1f}%)')


Loading from cache ...

✅ Dataset loaded
   Total cycles   : 6898
   Image shape    : (2, 64, 64)  (channels, H, W)
   Unique patients: 126

Class distribution:
   Normal    :  3642  (52.8%)
   Crackle   :  1864  (27.0%)
   Wheeze    :   886  (12.8%)
   Both      :   506  (7.3%)


## Step 6 — Patient Split + Oversampling + Class Weights


In [7]:
patients_arr    = np.array(patients_all)
unique_patients = sorted(set(patients_all))

valid_patients = [
    pid for pid in unique_patients
    if (patients_arr == pid).sum() >= MIN_CYCLES
]
print(f'Patients >= {MIN_CYCLES} cycles: {len(valid_patients)} / {len(unique_patients)}')

random.shuffle(valid_patients)
n_test         = max(1, int(0.2 * len(valid_patients)))
test_patients  = valid_patients[:n_test]
train_patients = valid_patients[n_test:]

test_mask = np.isin(patients_arr, test_patients)
X_test_g  = X_all[test_mask]   # (N_test, 2, 64, 64)
y_test_g  = y_all[test_mask]

# ── Per-client raw data ───────────────────────────────────────────────────────
client_data_raw = {}
for pid in train_patients:
    mask = patients_arr == pid
    Xp, yp = X_all[mask], y_all[mask]
    if len(Xp) < 2:
        continue
    split = max(1, int(0.8 * len(Xp)))
    client_data_raw[pid] = {
        'X_train': Xp[:split], 'y_train': yp[:split],
        'X_val':   Xp[split:], 'y_val':   yp[split:],
    }

NUM_CLIENTS = len(client_data_raw)
print(f'Active FL clients  : {NUM_CLIENTS}')
print(f'Global test cycles : {len(X_test_g)}')


# ── Per-channel normalization ─────────────────────────────────────────────────
# Compute mean/std per channel across all training images
X_train_concat = np.concatenate([v['X_train'] for v in client_data_raw.values()])
y_train_concat = np.concatenate([v['y_train'] for v in client_data_raw.values()])

ch_mean = X_train_concat.mean(axis=(0, 2, 3), keepdims=True)  # (1,2,1,1)
ch_std  = X_train_concat.std(axis=(0, 2, 3), keepdims=True) + 1e-8

def normalize_images(X):
    return (X - ch_mean) / ch_std

X_test_g = normalize_images(X_test_g)


# ── Oversampling helper ───────────────────────────────────────────────────────
def oversample_client(X, y, cap_ratio=2.5, majority_augment_frac=0.15, seed=SEED):
    rng = np.random.RandomState(seed)
    classes, counts = np.unique(y, return_counts=True)
    max_count = counts.max()
    X_parts, y_parts = [X], [y]
    for cls, count in zip(classes, counts):
        deficit = max_count - count
        n_extra = min(deficit, int(count * cap_ratio)) if deficit > 0 else int(count * majority_augment_frac)
        if n_extra <= 0:
            continue
        idx = np.where(y == cls)[0]
        sampled = rng.choice(idx, size=n_extra, replace=True)
        X_parts.append(X[sampled])
        y_parts.append(y[sampled])
    X_bal = np.concatenate(X_parts)
    y_bal = np.concatenate(y_parts)
    shuf  = rng.permutation(len(X_bal))
    return X_bal[shuf], y_bal[shuf]


# ── Build client_data with normalization + oversampling ───────────────────────
client_data = {}
for pid, d in client_data_raw.items():
    Xtr = normalize_images(d['X_train'])
    ytr = d['y_train']
    if len(np.unique(ytr)) > 1:
        Xtr, ytr = oversample_client(Xtr, ytr)
    client_data[pid] = {
        'X_train': Xtr, 'y_train': ytr,
        'X_val':   normalize_images(d['X_val']), 'y_val': d['y_val'],
    }

print('✓ Images normalized + clients oversampled')

# ── Class weights ─────────────────────────────────────────────────────────────
y_train_all = np.concatenate([v['y_train'] for v in client_data.values()])
class_counts  = np.bincount(y_train_all, minlength=NUM_CLASSES).astype(float)
class_weights = torch.tensor(
    (class_counts.sum() / (NUM_CLASSES * class_counts)) ** 2,
    dtype=torch.float32
)
class_weights = torch.clamp(class_weights, max=10.0)
print('\nClass weights:')
for name, w in zip(LABEL_NAMES, class_weights):
    print(f'   {name:10s}: {w:.3f}')


Patients >= 5 cycles: 124 / 126
Active FL clients  : 100
Global test cycles : 1185
✓ Images normalized + clients oversampled

Class weights:
   Normal    : 0.314
   Crackle   : 0.704
   Wheeze    : 2.578
   Both      : 6.252


## Step 7 — 2D CNN Model + Shared Utilities

`RespCNN` mirrors your teammate's CNN architecture but in PyTorch:  
3 Conv blocks (32→64→128 filters) + GlobalAvgPool + Dense head.  
Input: **(batch, 2, 64, 64)** — STFT and LogMel as two channels.


In [8]:
class RespCNN(nn.Module):
    """
    2D CNN on (2, 64, 64) spectrogram images.
    Mirrors teammate's build_baseline_cnn() in PyTorch.
    Channel 0 = STFT dB, Channel 1 = Log-Mel.
    """
    def __init__(self, num_classes=NUM_CLASSES, in_channels=N_CHANNELS):
        super().__init__()
        self.features = nn.Sequential(
            # Block 1
            nn.Conv2d(in_channels, 32, 3, padding=1),
            nn.BatchNorm2d(32), nn.ReLU(),
            nn.MaxPool2d(2), nn.Dropout2d(0.20),
            # Block 2
            nn.Conv2d(32, 64, 3, padding=1),
            nn.BatchNorm2d(64), nn.ReLU(),
            nn.MaxPool2d(2), nn.Dropout2d(0.25),
            # Block 3
            nn.Conv2d(64, 128, 3, padding=1),
            nn.BatchNorm2d(128), nn.ReLU(),
            nn.MaxPool2d(2), nn.Dropout2d(0.30),
        )
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(128, 64), nn.ReLU(), nn.Dropout(0.40),
            nn.Linear(64, num_classes)
        )

    def forward(self, x):
        return self.classifier(self.features(x))


# ── FL weight helpers ─────────────────────────────────────────────────────────
def get_weights(model):
    return [p.detach().cpu().numpy().copy() for p in model.parameters()]

def set_weights(model, weights):
    with torch.no_grad():
        for p, w in zip(model.parameters(), weights):
            p.copy_(torch.tensor(w))


# ── DataLoader helper ─────────────────────────────────────────────────────────
def make_loader(X, y, batch_size, shuffle=True):
    ds   = TensorDataset(
        torch.tensor(X, dtype=torch.float32),
        torch.tensor(y, dtype=torch.long)
    )
    drop = len(X) > batch_size
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle, drop_last=drop)


# ── Evaluation ────────────────────────────────────────────────────────────────
def evaluate(model, X, y, batch_size=64):
    model.eval()
    all_preds = []
    loader = DataLoader(
        TensorDataset(torch.tensor(X, dtype=torch.float32)),
        batch_size=batch_size, shuffle=False
    )
    with torch.no_grad():
        for (xb,) in loader:
            all_preds.append(model(xb).argmax(1).numpy())
    preds = np.concatenate(all_preds)
    acc   = (preds == y).mean()
    f1    = f1_score(y, preds, average='macro', zero_division=0)
    return acc, f1, preds


def predict_proba(model, X, batch_size=64):
    """Returns softmax probabilities (N, num_classes)."""
    model.eval()
    all_probs = []
    loader = DataLoader(
        TensorDataset(torch.tensor(X, dtype=torch.float32)),
        batch_size=batch_size, shuffle=False
    )
    with torch.no_grad():
        for (xb,) in loader:
            all_probs.append(torch.softmax(model(xb), dim=1).numpy())
    return np.concatenate(all_probs)


def print_report(y_true, y_pred, title='', label_names=LABEL_NAMES):
    if title:
        print(f'\n── {title} ──')
    print(classification_report(y_true, y_pred,
                                 target_names=label_names, zero_division=0))


# ── Focal Loss ────────────────────────────────────────────────────────────────
class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0, alpha=None, num_classes=NUM_CLASSES):
        super().__init__()
        self.gamma = gamma
        self.alpha = torch.ones(num_classes) if alpha is None else \
                     torch.tensor(alpha, dtype=torch.float32)

    def forward(self, logits, targets):
        probs    = torch.softmax(logits, dim=1)
        log_prob = torch.log_softmax(logits, dim=1)
        pt       = probs.gather(1, targets.unsqueeze(1)).squeeze(1)
        log_pt   = log_prob.gather(1, targets.unsqueeze(1)).squeeze(1)
        alpha_t  = self.alpha.to(logits.device)[targets]
        return (alpha_t * (1 - pt) ** self.gamma * (-log_pt)).mean()


# ── FedAvg aggregation ────────────────────────────────────────────────────────
def fedavg_aggregate(global_weights, client_weights, client_sizes):
    total = sum(client_sizes)
    return [
        sum((n / total) * cw[i] for cw, n in zip(client_weights, client_sizes))
        for i in range(len(global_weights))
    ]


# Sanity check
_m = RespCNN()
_x = torch.randn(4, N_CHANNELS, IMAGE_H, IMAGE_W)
assert _m(_x).shape == (4, NUM_CLASSES)
print(f'✓ RespCNN output : {_m(_x).shape}')
print(f'  Parameters     : {sum(p.numel() for p in _m.parameters()):,}')


✓ RespCNN output : torch.Size([4, 4])
  Parameters     : 101,924


## Step 8 — Centralized Baseline (2D CNN)


In [10]:
CENTRAL_EPOCHS  = 40
CENTRAL_PATIENCE = 8

X_train_all = np.concatenate([v['X_train'] for v in client_data.values()])
y_train_all_final = np.concatenate([v['y_train'] for v in client_data.values()])

central_model = RespCNN()
ce_weighted   = nn.CrossEntropyLoss(weight=class_weights)
opt_central   = torch.optim.Adam(central_model.parameters(), lr=1e-4)
scheduler_c   = torch.optim.lr_scheduler.ReduceLROnPlateau(
    opt_central, mode='max', factor=0.5, patience=3, min_lr=1e-6
)
loader_central = make_loader(X_train_all, y_train_all_final, BATCH_SIZE)

best_f1_c, best_state_c = 0, None
patience_counter = 0
history_central  = []

print(f'Training centralized CNN ({CENTRAL_EPOCHS} epochs, early stopping patience={CENTRAL_PATIENCE}) ...')
for epoch in range(1, CENTRAL_EPOCHS + 1):
    central_model.train()
    for xb, yb in loader_central:
        opt_central.zero_grad()
        ce_weighted(central_model(xb), yb).backward()
        opt_central.step()
    acc, f1, _ = evaluate(central_model, X_test_g, y_test_g)
    scheduler_c.step(f1)
    history_central.append({'epoch': epoch, 'acc': acc, 'f1': f1})
    if f1 > best_f1_c:
        best_f1_c    = f1
        best_state_c = copy.deepcopy(central_model.state_dict())
        patience_counter = 0
    else:
        patience_counter += 1
    if epoch % 10 == 0:
        print(f'  Epoch {epoch:3d}/{CENTRAL_EPOCHS} — Acc: {acc:.4f}  F1: {f1:.4f}  LR: {opt_central.param_groups[0]["lr"]:.2e}')
    if patience_counter >= CENTRAL_PATIENCE:
        print(f'  Early stopping at epoch {epoch}')
        break

central_model.load_state_dict(best_state_c)
torch.save(central_model.state_dict(), RESULTS_DIR / 'central_cnn.pt')

acc_c, f1_c, preds_c = evaluate(central_model, X_test_g, y_test_g)
print(f'\n✅ Centralized CNN — Acc: {acc_c:.4f}  Macro-F1: {f1_c:.4f}')
print_report(y_test_g, preds_c, 'Centralized CNN')


Training centralized CNN (40 epochs, early stopping patience=8) ...
  Epoch  10/40 — Acc: 0.1578  F1: 0.1532  LR: 1.00e-04
  Epoch  20/40 — Acc: 0.1544  F1: 0.1504  LR: 2.50e-05
  Epoch  30/40 — Acc: 0.1764  F1: 0.1713  LR: 1.25e-05
  Epoch  40/40 — Acc: 0.1890  F1: 0.1798  LR: 6.25e-06

✅ Centralized CNN — Acc: 0.1890  Macro-F1: 0.1798

── Centralized CNN ──
              precision    recall  f1-score   support

      Normal       0.00      0.00      0.00       575
     Crackle       0.24      0.43      0.30       267
      Wheeze       0.20      0.18      0.19       243
        Both       0.14      0.66      0.23       100

    accuracy                           0.19      1185
   macro avg       0.14      0.32      0.18      1185
weighted avg       0.11      0.19      0.13      1185



## Step 9 — FedAvg (2D CNN)


In [11]:
def local_train_fedavg(model, X_train, y_train, epochs, lr, batch_size, cw):
    if len(X_train) < batch_size:
        return 0
    model.train()
    loader = make_loader(X_train, y_train, batch_size)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    ce  = nn.CrossEntropyLoss(weight=cw)
    for _ in range(epochs):
        for xb, yb in loader:
            opt.zero_grad()
            ce(model(xb), yb).backward()
            opt.step()
    return len(X_train)


def run_fedavg(client_data, num_rounds, local_epochs, lr, batch_size, cw, label='FedAvg'):
    global_model = RespCNN()
    history = []
    for rnd in range(1, num_rounds + 1):
        global_w = get_weights(global_model)
        client_ws, client_ns = [], []
        for pid, data in client_data.items():
            local_model = RespCNN()
            set_weights(local_model, global_w)
            n = local_train_fedavg(local_model, data['X_train'], data['y_train'],
                                    local_epochs, lr, batch_size, cw)
            if n > 0:
                client_ws.append(get_weights(local_model))
                client_ns.append(n)
        if client_ws:
            set_weights(global_model, fedavg_aggregate(global_w, client_ws, client_ns))
        acc, f1, _ = evaluate(global_model, X_test_g, y_test_g)
        history.append({'round': rnd, 'acc': acc, 'f1': f1})
        print(f'  [{label}] Round {rnd:2d}/{num_rounds} — Acc: {acc:.4f}  F1: {f1:.4f}')
    return global_model, history


print('Running FedAvg (126 clients, 2D CNN) ...')
global_model_avg, history_avg = run_fedavg(
    client_data, NUM_ROUNDS, LOCAL_EPOCHS, LR, BATCH_SIZE, class_weights
)
torch.save(global_model_avg.state_dict(), RESULTS_DIR / 'fedavg_cnn.pt')
_, _, preds_avg = evaluate(global_model_avg, X_test_g, y_test_g)
print_report(y_test_g, preds_avg, 'FedAvg 126-client CNN')
print('\n✅ FedAvg complete.')


Running FedAvg (126 clients, 2D CNN) ...
  [FedAvg] Round  1/20 — Acc: 0.2051  F1: 0.0851
  [FedAvg] Round  2/20 — Acc: 0.2160  F1: 0.1026
  [FedAvg] Round  3/20 — Acc: 0.2759  F1: 0.1623
  [FedAvg] Round  4/20 — Acc: 0.3781  F1: 0.2165
  [FedAvg] Round  5/20 — Acc: 0.3882  F1: 0.2232
  [FedAvg] Round  6/20 — Acc: 0.4219  F1: 0.2336
  [FedAvg] Round  7/20 — Acc: 0.3983  F1: 0.2198
  [FedAvg] Round  8/20 — Acc: 0.4911  F1: 0.2386
  [FedAvg] Round  9/20 — Acc: 0.4135  F1: 0.2407
  [FedAvg] Round 10/20 — Acc: 0.4304  F1: 0.2268
  [FedAvg] Round 11/20 — Acc: 0.4287  F1: 0.2460
  [FedAvg] Round 12/20 — Acc: 0.4034  F1: 0.2363
  [FedAvg] Round 13/20 — Acc: 0.4118  F1: 0.2454
  [FedAvg] Round 14/20 — Acc: 0.4371  F1: 0.2564
  [FedAvg] Round 15/20 — Acc: 0.4160  F1: 0.2804
  [FedAvg] Round 16/20 — Acc: 0.4084  F1: 0.2725
  [FedAvg] Round 17/20 — Acc: 0.3409  F1: 0.2461
  [FedAvg] Round 18/20 — Acc: 0.3350  F1: 0.2403
  [FedAvg] Round 19/20 — Acc: 0.3316  F1: 0.2410
  [FedAvg] Round 20/20 — Acc

## Step 10 — FedProx μ Sweep (2D CNN)


In [12]:
def local_train_fedprox(model, global_weights, X_train, y_train,
                         epochs, lr, batch_size, mu, cw):
    if len(X_train) < batch_size:
        return 0
    model.train()
    loader = make_loader(X_train, y_train, batch_size)
    opt    = torch.optim.Adam(model.parameters(), lr=lr)
    ce     = nn.CrossEntropyLoss(weight=cw)
    global_tensors = [torch.tensor(w, dtype=torch.float32) for w in global_weights]
    for _ in range(epochs):
        for xb, yb in loader:
            opt.zero_grad()
            loss = ce(model(xb), yb)
            prox = sum(((p - gw)**2).sum()
                       for p, gw in zip(model.parameters(), global_tensors))
            (loss + (mu / 2) * prox).backward()
            opt.step()
    return len(X_train)


def run_fedprox(client_data, num_rounds, local_epochs, lr, batch_size, mu, cw):
    global_model = RespCNN()
    history = []
    for rnd in range(1, num_rounds + 1):
        global_w = get_weights(global_model)
        client_ws, client_ns = [], []
        for pid, data in client_data.items():
            local_model = RespCNN()
            set_weights(local_model, global_w)
            n = local_train_fedprox(local_model, global_w,
                                     data['X_train'], data['y_train'],
                                     local_epochs, lr, batch_size, mu, cw)
            if n > 0:
                client_ws.append(get_weights(local_model))
                client_ns.append(n)
        if client_ws:
            set_weights(global_model, fedavg_aggregate(global_w, client_ws, client_ns))
        acc, f1, _ = evaluate(global_model, X_test_g, y_test_g)
        history.append({'round': rnd, 'acc': acc, 'f1': f1})
        print(f'  [FedProx μ={mu}] Round {rnd:2d}/{num_rounds} — Acc: {acc:.4f}  F1: {f1:.4f}')
    return global_model, history


prox_results = {}
for mu_val in MU_SWEEP:
    print(f'\nRunning FedProx μ={mu_val} ...')
    mdl, hist = run_fedprox(
        client_data, NUM_ROUNDS, LOCAL_EPOCHS, LR, BATCH_SIZE, mu_val, class_weights
    )
    acc, f1, preds = evaluate(mdl, X_test_g, y_test_g)
    prox_results[mu_val] = {'model': mdl, 'history': hist,
                             'acc': acc, 'f1': f1, 'preds': preds}
    torch.save(mdl.state_dict(), RESULTS_DIR / f'fedprox_mu{mu_val}_cnn.pt')
    print_report(y_test_g, preds, f'FedProx μ={mu_val}')

best_mu           = max(prox_results, key=lambda m: prox_results[m]['f1'])
global_model_prox = prox_results[best_mu]['model']
history_prox      = prox_results[best_mu]['history']
print(f'\n✅ FedProx complete. Best μ={best_mu}  F1={prox_results[best_mu]["f1"]:.4f}')



Running FedProx μ=0.1 ...
  [FedProx μ=0.1] Round  1/20 — Acc: 0.4852  F1: 0.1634
  [FedProx μ=0.1] Round  2/20 — Acc: 0.4852  F1: 0.1634
  [FedProx μ=0.1] Round  3/20 — Acc: 0.4852  F1: 0.1634
  [FedProx μ=0.1] Round  4/20 — Acc: 0.4852  F1: 0.1634
  [FedProx μ=0.1] Round  5/20 — Acc: 0.4852  F1: 0.1634
  [FedProx μ=0.1] Round  6/20 — Acc: 0.4852  F1: 0.1634
  [FedProx μ=0.1] Round  7/20 — Acc: 0.4852  F1: 0.1634
  [FedProx μ=0.1] Round  8/20 — Acc: 0.4852  F1: 0.1634
  [FedProx μ=0.1] Round  9/20 — Acc: 0.4852  F1: 0.1634
  [FedProx μ=0.1] Round 10/20 — Acc: 0.4852  F1: 0.1634
  [FedProx μ=0.1] Round 11/20 — Acc: 0.4852  F1: 0.1634
  [FedProx μ=0.1] Round 12/20 — Acc: 0.4852  F1: 0.1634
  [FedProx μ=0.1] Round 13/20 — Acc: 0.4852  F1: 0.1634
  [FedProx μ=0.1] Round 14/20 — Acc: 0.4852  F1: 0.1634
  [FedProx μ=0.1] Round 15/20 — Acc: 0.4852  F1: 0.1634
  [FedProx μ=0.1] Round 16/20 — Acc: 0.4852  F1: 0.1634
  [FedProx μ=0.1] Round 17/20 — Acc: 0.4852  F1: 0.1634
  [FedProx μ=0.1] Rou

## Step 11 — 10-Client Grouping Ablation


In [13]:
N_GROUPS = 10
pids     = sorted(client_data.keys())
groups   = np.array_split(pids, N_GROUPS)

client_data_10 = {}
for g_idx, group in enumerate(groups):
    if len(group) == 0:
        continue
    client_data_10[f'group_{g_idx}'] = {
        'X_train': np.concatenate([client_data[p]['X_train'] for p in group]),
        'y_train': np.concatenate([client_data[p]['y_train'] for p in group]),
        'X_val':   np.concatenate([client_data[p]['X_val']   for p in group]),
        'y_val':   np.concatenate([client_data[p]['y_val']   for p in group]),
    }

print(f'10-client grouping: {len(client_data_10)} super-clients')
print('Running FedAvg on 10-client grouping ...')
global_model_10, history_10 = run_fedavg(
    client_data_10, NUM_ROUNDS, LOCAL_EPOCHS, LR, BATCH_SIZE, class_weights, label='FedAvg-10'
)
acc_10, f1_10, preds_10 = evaluate(global_model_10, X_test_g, y_test_g)
print_report(y_test_g, preds_10, 'FedAvg 10-client')
print(f'✅ 10-client — Acc: {acc_10:.4f}  F1: {f1_10:.4f}')


10-client grouping: 10 super-clients
Running FedAvg on 10-client grouping ...
  [FedAvg-10] Round  1/20 — Acc: 0.1983  F1: 0.0844
  [FedAvg-10] Round  2/20 — Acc: 0.1249  F1: 0.0959
  [FedAvg-10] Round  3/20 — Acc: 0.1536  F1: 0.1185
  [FedAvg-10] Round  4/20 — Acc: 0.2051  F1: 0.0859
  [FedAvg-10] Round  5/20 — Acc: 0.2025  F1: 0.1132
  [FedAvg-10] Round  6/20 — Acc: 0.1797  F1: 0.1544
  [FedAvg-10] Round  7/20 — Acc: 0.2143  F1: 0.1454
  [FedAvg-10] Round  8/20 — Acc: 0.2118  F1: 0.1365
  [FedAvg-10] Round  9/20 — Acc: 0.2143  F1: 0.2065
  [FedAvg-10] Round 10/20 — Acc: 0.2127  F1: 0.1690
  [FedAvg-10] Round 11/20 — Acc: 0.2186  F1: 0.1796
  [FedAvg-10] Round 12/20 — Acc: 0.2186  F1: 0.1602
  [FedAvg-10] Round 13/20 — Acc: 0.2160  F1: 0.1563
  [FedAvg-10] Round 14/20 — Acc: 0.1637  F1: 0.1453
  [FedAvg-10] Round 15/20 — Acc: 0.2194  F1: 0.2089
  [FedAvg-10] Round 16/20 — Acc: 0.1781  F1: 0.1566
  [FedAvg-10] Round 17/20 — Acc: 0.1511  F1: 0.1446
  [FedAvg-10] Round 18/20 — Acc: 0.128

## Step 12 — Dedicated Stage 1 Binary CNN

**Key fix from v3**: Stage 1 is now a dedicated binary CNN trained only for Normal vs Abnormal.  
v3 reused the FedAvg global model which was never optimized for this binary decision.  
Threshold is auto-selected from validation sweep (mirrors teammate Cell 54).


In [14]:
print('=== STAGE 1 — DEDICATED BINARY CNN ===')

# Binary labels: 0=Normal, 1=Abnormal
y_binary_train = (y_train_all_final != 0).astype(np.int64)
y_binary_test  = (y_test_g != 0).astype(np.int64)

# Validation split for threshold selection (20% of training data)
val_split   = int(0.8 * len(X_train_all))
X_s1_train  = X_train_all[:val_split]
y_s1_train  = y_binary_train[:val_split]
X_s1_val    = X_train_all[val_split:]
y_s1_val    = y_binary_train[val_split:]

# Binary class weights
bin_counts  = np.bincount(y_s1_train).astype(float)
bin_weights = torch.tensor(
    [bin_counts.sum() / (2 * bin_counts[0]),
     bin_counts.sum() / (2 * bin_counts[1])],
    dtype=torch.float32
)

stage1_model  = RespCNN(num_classes=2)
opt_s1        = torch.optim.Adam(stage1_model.parameters(), lr=1e-4)
scheduler_s1  = torch.optim.lr_scheduler.ReduceLROnPlateau(
    opt_s1, mode='max', factor=0.5, patience=3, min_lr=1e-6
)
ce_binary     = nn.CrossEntropyLoss(weight=bin_weights)
loader_s1     = make_loader(X_s1_train, y_s1_train, BATCH_SIZE)

best_s1_f1    = 0
best_s1_state = None
patience_s1   = 0
PATIENCE_S1   = 8

print(f'Training Stage 1 binary CNN (patience={PATIENCE_S1}) ...')
for epoch in range(1, 81):
    stage1_model.train()
    for xb, yb in loader_s1:
        opt_s1.zero_grad()
        ce_binary(stage1_model(xb), yb).backward()
        opt_s1.step()
    # Evaluate on validation split
    stage1_model.eval()
    s1_probs_val = predict_proba(stage1_model, X_s1_val)
    s1_preds_val = s1_probs_val[:, 1] >= 0.5
    f1_val = f1_score(y_s1_val, s1_preds_val, average='macro', zero_division=0)
    scheduler_s1.step(f1_val)
    if f1_val > best_s1_f1:
        best_s1_f1    = f1_val
        best_s1_state = copy.deepcopy(stage1_model.state_dict())
        patience_s1   = 0
    else:
        patience_s1 += 1
    if epoch % 10 == 0:
        print(f'  Epoch {epoch:2d}/80 — Val Binary F1: {f1_val:.4f}')
    if patience_s1 >= PATIENCE_S1:
        print(f'  Early stopping at epoch {epoch}')
        break

stage1_model.load_state_dict(best_s1_state)
torch.save(stage1_model.state_dict(), RESULTS_DIR / 'stage1_binary_cnn.pt')

# Get Stage 1 probabilities on test set
s1_test_probs  = predict_proba(stage1_model, X_test_g)  # (N, 2)
s1_abnorm_prob = s1_test_probs[:, 1]   # P(Abnormal)
s1_normal_prob = s1_test_probs[:, 0]   # P(Normal)

# ── Auto threshold selection from validation sweep ────────────────────────────
# Mirror of teammate Cell 54
print('\nThreshold sweep on validation set:')
print(f'{"Threshold":>10} {"Routed":>8} {"Recall_abn":>12} {"Val_F1":>10}')

true_abn     = y_test_g != 0
thresh_results = {}

for thresh in np.arange(0.30, 0.71, 0.05):
    thresh = round(thresh, 2)
    s1p    = (s1_abnorm_prob >= thresh).astype(int)
    routed = ((s1p == 1) & true_abn).sum()
    recall = routed / true_abn.sum()
    f1_bin = f1_score(y_binary_test, s1p, average='macro', zero_division=0)
    thresh_results[thresh] = {'routed': routed, 'recall': recall, 'f1': f1_bin}
    print(f'{thresh:>10.2f} {routed:>8} {recall:>12.4f} {f1_bin:>10.4f}')

# Pick best threshold by binary macro-F1
STAGE1_THRESHOLD = max(thresh_results, key=lambda t: thresh_results[t]['f1'])
stage1_preds     = (s1_abnorm_prob >= STAGE1_THRESHOLD).astype(int)

print(f'\n✅ Best threshold : {STAGE1_THRESHOLD}')
print(f'   Routing recall : {thresh_results[STAGE1_THRESHOLD]["recall"]:.4f}')
print(f'   Binary F1      : {thresh_results[STAGE1_THRESHOLD]["f1"]:.4f}')
print(f'   Samples → Stage 2 : {stage1_preds.sum()}')


=== STAGE 1 — DEDICATED BINARY CNN ===
Training Stage 1 binary CNN (patience=8) ...
  Epoch 10/80 — Val Binary F1: 0.5985
  Early stopping at epoch 11

Threshold sweep on validation set:
 Threshold   Routed   Recall_abn     Val_F1
      0.30      610       1.0000     0.3398
      0.35      609       0.9984     0.3509
      0.40      597       0.9787     0.4098
      0.45      568       0.9311     0.5397
      0.50      497       0.8148     0.6217
      0.55      377       0.6180     0.6639
      0.60      217       0.3557     0.5726
      0.65       65       0.1066     0.4265
      0.70        1       0.0016     0.3285

✅ Best threshold : 0.55
   Routing recall : 0.6180
   Binary F1      : 0.6639
   Samples → Stage 2 : 542


## Step 13 — Stage 2 CNN (Crackle / Wheeze / Both)


In [15]:
STAGE2_CLASSES     = ['Crackle', 'Wheeze', 'Both']
STAGE2_NUM_CLASSES = 3
STAGE2_EPOCHS      = 80
PATIENCE_S2        = 8

# Collect abnormal training cycles (labels 1,2,3 → remap to 0,1,2)
abn_mask_train = y_train_all_final != 0
X_abn = X_train_all[abn_mask_train]
y_abn = y_train_all_final[abn_mask_train] - 1   # 1,2,3 → 0,1,2

# Oversample with cap_ratio=2.5 (default — will be tuned in ablation)
X_abn_bal, y_abn_bal = oversample_client(X_abn, y_abn, cap_ratio=2.5)

print(f'Stage 2 training cycles (balanced): {len(X_abn_bal)}')
for i, name in enumerate(STAGE2_CLASSES):
    print(f'  {name:10s}: {(y_abn_bal==i).sum()}')

# Focal loss weights for Stage 2
s2_counts = np.bincount(y_abn_bal, minlength=STAGE2_NUM_CLASSES).astype(float)
s2_alpha  = (s2_counts.sum() / (STAGE2_NUM_CLASSES * s2_counts))

# Stage 2 validation split
s2_val_split = int(0.8 * len(X_abn_bal))
X_s2_val = X_abn_bal[s2_val_split:]
y_s2_val = y_abn_bal[s2_val_split:]
X_s2_tr  = X_abn_bal[:s2_val_split]
y_s2_tr  = y_abn_bal[:s2_val_split]

stage2_model  = RespCNN(num_classes=STAGE2_NUM_CLASSES)
focal_s2      = FocalLoss(gamma=2.0, alpha=s2_alpha, num_classes=STAGE2_NUM_CLASSES)
opt_s2        = torch.optim.Adam(stage2_model.parameters(), lr=LR)
scheduler_s2  = torch.optim.lr_scheduler.ReduceLROnPlateau(
    opt_s2, mode='max', factor=0.5, patience=3, min_lr=1e-6
)
loader_s2     = make_loader(X_s2_tr, y_s2_tr, BATCH_SIZE)

best_s2_f1    = 0
best_s2_state = None
patience_s2   = 0

print(f'\nTraining Stage 2 CNN ({STAGE2_EPOCHS} epochs) ...')
for epoch in range(1, STAGE2_EPOCHS + 1):
    stage2_model.train()
    for xb, yb in loader_s2:
        opt_s2.zero_grad()
        focal_s2(stage2_model(xb), yb).backward()
        opt_s2.step()
    stage2_model.eval()
    _, f1_s2_val, _ = evaluate(stage2_model, X_s2_val, y_s2_val)
    scheduler_s2.step(f1_s2_val)
    if f1_s2_val > best_s2_f1:
        best_s2_f1    = f1_s2_val
        best_s2_state = copy.deepcopy(stage2_model.state_dict())
        patience_s2   = 0
    else:
        patience_s2 += 1
    if epoch % 10 == 0:
        print(f'  Epoch {epoch:3d}/{STAGE2_EPOCHS} — Val F1: {f1_s2_val:.4f}')
    if patience_s2 >= PATIENCE_S2:
        print(f'  Early stopping at epoch {epoch}')
        break

stage2_model.load_state_dict(best_s2_state)
torch.save(stage2_model.state_dict(), RESULTS_DIR / 'stage2_cnn.pt')
print('\n✅ Stage 2 CNN trained.')


# ── Initial cascade evaluation ────────────────────────────────────────────────
def run_cascade(s1_preds, s2_model, X_test, y_test, label_names=LABEL_NAMES):
    final_preds = np.zeros(len(y_test), dtype=int)
    abn_idx = np.where(s1_preds == 1)[0]
    if len(abn_idx) > 0:
        s2_model.eval()
        with torch.no_grad():
            s2p = s2_model(
                torch.tensor(X_test[abn_idx], dtype=torch.float32)
            ).argmax(1).numpy()
        final_preds[abn_idx] = s2p + 1
    acc = (final_preds == y_test).mean()
    f1  = f1_score(y_test, final_preds, average='macro', zero_division=0)
    return final_preds, acc, f1


final_preds_orig, cascade_acc_orig, cascade_f1_orig = run_cascade(
    stage1_preds, stage2_model, X_test_g, y_test_g
)
print(f'\n✅ ORIGINAL CASCADE — Acc: {cascade_acc_orig:.4f}  Macro-F1: {cascade_f1_orig:.4f}')
print_report(y_test_g, final_preds_orig, 'Original Two-Stage Cascade')


Stage 2 training cycles (balanced): 6552
  Crackle   : 2392
  Wheeze    : 2080
  Both      : 2080

Training Stage 2 CNN (80 epochs) ...
  Epoch  10/80 — Val F1: 0.5240
  Epoch  20/80 — Val F1: 0.5390
  Epoch  30/80 — Val F1: 0.6158
  Epoch  40/80 — Val F1: 0.6243
  Epoch  50/80 — Val F1: 0.6476
  Epoch  60/80 — Val F1: 0.6682
  Epoch  70/80 — Val F1: 0.6721
  Early stopping at epoch 73

✅ Stage 2 CNN trained.

✅ ORIGINAL CASCADE — Acc: 0.5148  Macro-F1: 0.4100

── Original Two-Stage Cascade ──
              precision    recall  f1-score   support

      Normal       0.64      0.71      0.67       575
     Crackle       0.45      0.43      0.44       267
      Wheeze       0.54      0.23      0.33       243
        Both       0.15      0.28      0.20       100

    accuracy                           0.51      1185
   macro avg       0.45      0.41      0.41      1185
weighted avg       0.54      0.51      0.51      1185



## Step 14 — Full Ablation Study

Each ablation sweeps one parameter, auto-selects the best value, and passes it forward.  
Run in order: Ablation 1 → 2 → 3 → 4 → Final retrain.


In [16]:
# ── Ablation 1 — Stage 1 Threshold ───────────────────────────────────────────
print('ABLATION 1 — Stage 1 Threshold Sweep')
print(f'{"Threshold":>10} {"→Normal":>8} {"→Stage2":>8} {"Acc":>8} {"Cascade F1":>12}')

abl1_results = {}
for thresh in np.arange(0.25, 0.76, 0.05):
    thresh = round(thresh, 2)
    s1p    = (s1_abnorm_prob >= thresh).astype(int)
    fp, acc_t, f1_t = run_cascade(s1p, stage2_model, X_test_g, y_test_g)
    abl1_results[thresh] = {'acc': acc_t, 'f1': f1_t, 'preds': s1p}
    print(f'{thresh:>10.2f} {(s1p==0).sum():>8} {(s1p==1).sum():>8} {acc_t:>8.4f} {f1_t:>12.4f}')

BEST_THRESHOLD = max(abl1_results, key=lambda t: abl1_results[t]['f1'])
stage1_preds   = abl1_results[BEST_THRESHOLD]['preds']
print(f'\n✅ Best threshold : {BEST_THRESHOLD}  (Cascade F1={abl1_results[BEST_THRESHOLD]["f1"]:.4f})')
print(f'✓ STAGE1_THRESHOLD updated to {BEST_THRESHOLD}')


ABLATION 1 — Stage 1 Threshold Sweep
 Threshold  →Normal  →Stage2      Acc   Cascade F1
      0.25        0     1185   0.2886       0.2597
      0.30        0     1185   0.2886       0.2597
      0.35        7     1178   0.2937       0.2660
      0.40       55     1130   0.3165       0.2952
      0.45      184     1001   0.3823       0.3589
      0.50      371      814   0.4464       0.3945
      0.55      643      542   0.5148       0.4100
      0.60      881      304   0.4996       0.3386
      0.65     1103       82   0.4987       0.2438
      0.70     1184        1   0.4861       0.1684
      0.75     1185        0   0.4852       0.1634

✅ Best threshold : 0.55  (Cascade F1=0.4100)
✓ STAGE1_THRESHOLD updated to 0.55


In [17]:
# ── Ablation 2 — cap_ratio (oversampling) ─────────────────────────────────────
print('ABLATION 2 — Oversampling cap_ratio Sweep')
print(f'{"cap_ratio":>10} {"Train samples":>14} {"S2 Val F1":>12} {"Cascade F1":>12}')

abl2_results = {}
for cap in [1.5, 2.5, 4.0, 6.0, 8.0]:
    X_b, y_b  = oversample_client(X_abn, y_abn, cap_ratio=cap)
    s2_b      = np.bincount(y_b, minlength=3).astype(float)
    alpha_b   = s2_b.sum() / (3 * s2_b)
    s2_v_split = int(0.8 * len(X_b))
    m  = RespCNN(num_classes=3)
    fl = FocalLoss(gamma=2.0, alpha=alpha_b, num_classes=3)
    o  = torch.optim.Adam(m.parameters(), lr=LR)
    ld = make_loader(X_b[:s2_v_split], y_b[:s2_v_split], BATCH_SIZE)
    best_f1_b, best_state_b, pat_b = 0, None, 0
    for ep in range(60):
        m.train()
        for xb, yb in ld:
            o.zero_grad(); fl(m(xb), yb).backward(); o.step()
        _, f1_v, _ = evaluate(m, X_b[s2_v_split:], y_b[s2_v_split:])
        if f1_v > best_f1_b:
            best_f1_b = f1_v; best_state_b = copy.deepcopy(m.state_dict()); pat_b = 0
        else:
            pat_b += 1
        if pat_b >= 8: break
    m.load_state_dict(best_state_b)
    _, _, f1_cas = run_cascade(stage1_preds, m, X_test_g, y_test_g)
    abl2_results[cap] = {'f1_val': best_f1_b, 'f1_cascade': f1_cas,
                          'X_bal': X_b, 'y_bal': y_b}
    print(f'{cap:>10.1f} {len(X_b):>14} {best_f1_b:>12.4f} {f1_cas:>12.4f}')

BEST_CAP_RATIO = max(abl2_results, key=lambda c: abl2_results[c]['f1_cascade'])
X_abn_bal = abl2_results[BEST_CAP_RATIO]['X_bal']
y_abn_bal = abl2_results[BEST_CAP_RATIO]['y_bal']
print(f'\n✅ Best cap_ratio : {BEST_CAP_RATIO}  (Cascade F1={abl2_results[BEST_CAP_RATIO]["f1_cascade"]:.4f})')
print(f'✓ X_abn_bal updated with cap_ratio={BEST_CAP_RATIO}')


ABLATION 2 — Oversampling cap_ratio Sweep
 cap_ratio  Train samples    S2 Val F1   Cascade F1
       1.5           6217       0.7803       0.4038
       2.5           6552       0.5235       0.3624
       4.0           6552       0.5304       0.3562
       6.0           6552       0.7600       0.3982
       8.0           6552       0.5968       0.3976

✅ Best cap_ratio : 1.5  (Cascade F1=0.4038)
✓ X_abn_bal updated with cap_ratio=1.5


In [ ]:
# ── Ablation 3 — Focal Loss Gamma ─────────────────────────────────────────────
print('ABLATION 3 — Focal Loss Gamma Sweep')
print(f'{"gamma":>8} {"S2 Val F1":>12} {"Cascade F1":>12}')

abl3_results = {}
s2_counts_best = np.bincount(y_abn_bal, minlength=3).astype(float)
s2_alpha_best  = s2_counts_best.sum() / (3 * s2_counts_best)
s2_v_split     = int(0.8 * len(X_abn_bal))

for gamma_val in [0.5, 1.0, 1.5, 2.0, 2.5, 3.0]:
    m  = RespCNN(num_classes=3)
    fl = FocalLoss(gamma=gamma_val, alpha=s2_alpha_best, num_classes=3)
    o  = torch.optim.Adam(m.parameters(), lr=LR)
    ld = make_loader(X_abn_bal[:s2_v_split], y_abn_bal[:s2_v_split], BATCH_SIZE)
    best_f1_g, best_state_g, pat_g = 0, None, 0
    for ep in range(80):
        m.train()
        for xb, yb in ld:
            o.zero_grad(); fl(m(xb), yb).backward(); o.step()
        _, f1_v, _ = evaluate(m, X_abn_bal[s2_v_split:], y_abn_bal[s2_v_split:])
        if f1_v > best_f1_g:
            best_f1_g = f1_v; best_state_g = copy.deepcopy(m.state_dict()); pat_g = 0
        else:
            pat_g += 1
        if pat_g >= 8: break
    m.load_state_dict(best_state_g)
    _, _, f1_cas = run_cascade(stage1_preds, m, X_test_g, y_test_g)
    abl3_results[gamma_val] = {'f1_val': best_f1_g, 'f1_cascade': f1_cas}
    print(f'{gamma_val:>8.1f} {best_f1_g:>12.4f} {f1_cas:>12.4f}')

BEST_GAMMA = max(abl3_results, key=lambda g: abl3_results[g]['f1_cascade'])
print(f'\n✅ Best gamma     : {BEST_GAMMA}  (Cascade F1={abl3_results[BEST_GAMMA]["f1_cascade"]:.4f})')
print(f'✓ gamma updated to {BEST_GAMMA}')


ABLATION 3 — Focal Loss Gamma Sweep
   gamma    S2 Val F1   Cascade F1
     0.5       0.8745       0.4079
     1.0       0.8790       0.3925
     1.5       0.8795       0.4133
     2.0       0.5258       0.3572
     2.5       0.6330       0.4210


In [ ]:
# ── Ablation 4 — Stage 2 Learning Rate ────────────────────────────────────────
print('ABLATION 4 — Stage 2 Learning Rate Sweep')
print(f'{"LR":>10} {"S2 Val F1":>12} {"Cascade F1":>12}')

abl4_results = {}
s2_v_split   = int(0.8 * len(X_abn_bal))

for lr_val in [1e-4, 3e-4, 5e-4, 1e-3, 3e-3]:
    m  = RespCNN(num_classes=3)
    fl = FocalLoss(gamma=BEST_GAMMA, alpha=s2_alpha_best, num_classes=3)
    o  = torch.optim.Adam(m.parameters(), lr=lr_val)
    sc = torch.optim.lr_scheduler.ReduceLROnPlateau(o, mode='max', factor=0.5, patience=3, min_lr=1e-7)
    ld = make_loader(X_abn_bal[:s2_v_split], y_abn_bal[:s2_v_split], BATCH_SIZE)
    best_f1_l, best_state_l, pat_l = 0, None, 0
    for ep in range(80):
        m.train()
        for xb, yb in ld:
            o.zero_grad(); fl(m(xb), yb).backward(); o.step()
        _, f1_v, _ = evaluate(m, X_abn_bal[s2_v_split:], y_abn_bal[s2_v_split:])
        sc.step(f1_v)
        if f1_v > best_f1_l:
            best_f1_l = f1_v; best_state_l = copy.deepcopy(m.state_dict()); pat_l = 0
        else:
            pat_l += 1
        if pat_l >= 8: break
    m.load_state_dict(best_state_l)
    _, _, f1_cas = run_cascade(stage1_preds, m, X_test_g, y_test_g)
    abl4_results[lr_val] = {'f1_val': best_f1_l, 'f1_cascade': f1_cas}
    print(f'{lr_val:>10} {best_f1_l:>12.4f} {f1_cas:>12.4f}')

BEST_LR = max(abl4_results, key=lambda l: abl4_results[l]['f1_cascade'])
print(f'\n✅ Best LR        : {BEST_LR}  (Cascade F1={abl4_results[BEST_LR]["f1_cascade"]:.4f})')
print(f'✓ LR updated to {BEST_LR}')


## Step 15 — Final Retrain with Best Ablation Parameters


In [ ]:
print('=== FINAL RETRAIN WITH BEST ABLATION PARAMETERS ===')
print(f'  Threshold  : {BEST_THRESHOLD}')
print(f'  cap_ratio  : {BEST_CAP_RATIO}')
print(f'  gamma      : {BEST_GAMMA}')
print(f'  LR         : {BEST_LR}')

# Rebuild Stage 2 training data with best cap_ratio
X_abn_final, y_abn_final = oversample_client(X_abn, y_abn, cap_ratio=BEST_CAP_RATIO)
s2_counts_f = np.bincount(y_abn_final, minlength=3).astype(float)
s2_alpha_f  = s2_counts_f.sum() / (3 * s2_counts_f)
s2_v_split  = int(0.8 * len(X_abn_final))

stage2_final  = RespCNN(num_classes=3)
focal_final   = FocalLoss(gamma=BEST_GAMMA, alpha=s2_alpha_f, num_classes=3)
opt_final     = torch.optim.Adam(stage2_final.parameters(), lr=BEST_LR)
sched_final   = torch.optim.lr_scheduler.ReduceLROnPlateau(
    opt_final, mode='max', factor=0.5, patience=3, min_lr=1e-7
)
loader_final  = make_loader(X_abn_final[:s2_v_split], y_abn_final[:s2_v_split], BATCH_SIZE)

best_f2_f, best_state_f2, pat_f2 = 0, None, 0

print('\nTraining final Stage 2 CNN (max 100 epochs) ...')
for epoch in range(1, 101):
    stage2_final.train()
    for xb, yb in loader_final:
        opt_final.zero_grad()
        focal_final(stage2_final(xb), yb).backward()
        opt_final.step()
    _, f1_v, _ = evaluate(stage2_final, X_abn_final[s2_v_split:], y_abn_final[s2_v_split:])
    sched_final.step(f1_v)
    if f1_v > best_f2_f:
        best_f2_f = f1_v; best_state_f2 = copy.deepcopy(stage2_final.state_dict()); pat_f2 = 0
    else:
        pat_f2 += 1
    if epoch % 20 == 0:
        print(f'  Epoch {epoch:3d}/100 — Val F1: {f1_v:.4f}')
    if pat_f2 >= 10:
        print(f'  Early stopping at epoch {epoch}')
        break

stage2_final.load_state_dict(best_state_f2)
torch.save(stage2_final.state_dict(), RESULTS_DIR / 'stage2_final_tuned.pt')

# Final Stage 1 predictions with best threshold
stage1_preds_final = (s1_abnorm_prob >= BEST_THRESHOLD).astype(int)

final_preds_tuned, cascade_acc_tuned, cascade_f1_tuned = run_cascade(
    stage1_preds_final, stage2_final, X_test_g, y_test_g
)

print(f'\n✅ TUNED CASCADE — Acc: {cascade_acc_tuned:.4f}  Macro-F1: {cascade_f1_tuned:.4f}')
print_report(y_test_g, final_preds_tuned, 'Final Tuned Two-Stage Cascade')


## Step 16 — Patient Fairness Analysis


In [ ]:
fairness_rows = []
for pid in test_patients:
    mask = patients_arr == pid
    Xp   = normalize_images(X_all[mask])
    yp   = y_all[mask]
    if len(Xp) == 0:
        continue
    acc_p, f1_p, _ = evaluate(global_model_avg, Xp, yp)
    fairness_rows.append({
        'patient': pid, 'n_cycles': len(yp),
        'accuracy': round(float(acc_p), 4),
        'macro_f1': round(float(f1_p), 4),
        'classes_present': sorted(np.unique(yp).tolist())
    })

df_fair = pd.DataFrame(fairness_rows).sort_values('macro_f1')
print('Patient-level fairness (FedAvg global model):')
print(df_fair.to_string(index=False))
low_f1 = df_fair[df_fair['macro_f1'] < 0.2]
print(f'\n⚠  {len(low_f1)} patients with Macro-F1 < 0.2' if len(low_f1)
      else '\n✓ No patients with Macro-F1 < 0.2')
df_fair.to_csv(RESULTS_DIR / 'patient_fairness.csv', index=False)
print('✓ Saved patient_fairness.csv')


## Step 17 — Full Results Comparison


In [ ]:
# ── Convergence curves ────────────────────────────────────────────────────────
rounds = [h['round'] for h in history_avg]
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
styles = [
    (history_avg,  'FedAvg (126-client)', 'b-o'),
    (history_prox, f'FedProx μ={best_mu}', 'r-s'),
    (history_10,   'FedAvg (10-client)',  'g-^'),
]
for ax, metric, ylabel in zip(axes, ['acc', 'f1'], ['Accuracy', 'Macro-F1']):
    for hist, label, style in styles:
        ax.plot(rounds, [h[metric] for h in hist], style, label=label,
                linewidth=2, markersize=5)
    baseline = acc_c if metric == 'acc' else f1_c
    ax.axhline(baseline, color='k', linestyle='--', linewidth=1.5,
               label=f'Centralized ({baseline:.3f})')
    ax.axhline(cascade_f1_tuned if metric == 'f1' else cascade_acc_tuned,
               color='purple', linestyle=':', linewidth=1.5,
               label=f'Tuned Cascade ({cascade_f1_tuned:.3f})')
    ax.set_title(ylabel, fontsize=13)
    ax.set_xlabel('Round')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)
plt.suptitle('Convergence — All Methods (2D CNN)', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'convergence_all_cnn.png', dpi=150, bbox_inches='tight')
plt.show()
print('✓ Saved convergence_all_cnn.png')


In [ ]:
# ── Confusion matrices ────────────────────────────────────────────────────────
_, _, preds_prox_best = evaluate(global_model_prox, X_test_g, y_test_g)

fig, axes = plt.subplots(1, 4, figsize=(22, 5))
combos = [
    (preds_c,           'Centralized CNN'),
    (preds_avg,         'FedAvg 126-client'),
    (preds_prox_best,   f'FedProx μ={best_mu}'),
    (final_preds_tuned, 'Tuned Cascade'),
]
for ax, (preds, title) in zip(axes, combos):
    cm = confusion_matrix(y_test_g, preds)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=LABEL_NAMES, yticklabels=LABEL_NAMES)
    ax.set_title(title, fontsize=11)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('True')
    ax.tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'confusion_matrices_cnn.png', dpi=150)
plt.show()
print('✓ Saved confusion_matrices_cnn.png')


In [ ]:
# ── Before vs After Ablation Comparison ──────────────────────────────────────
print('=' * 65)
print('       ABLATION STUDY — BEFORE vs AFTER')
print('=' * 65)

print('\n📌 PARAMETERS:')
print(f'  Threshold  : 0.55 (auto)        → {BEST_THRESHOLD}')
print(f'  cap_ratio  : 2.5                → {BEST_CAP_RATIO}')
print(f'  gamma      : 2.0                → {BEST_GAMMA}')
print(f'  LR         : {LR}               → {BEST_LR}')

print('\n📊 OVERALL METRICS:')
print(f'  {"Metric":>12}   {"Original":>10}   {"Tuned":>10}   {"Change":>10}')
print(f'  {"-"*50}')
print(f'  {"Accuracy":>12}   {cascade_acc_orig:>10.4f}   {cascade_acc_tuned:>10.4f}   {cascade_acc_tuned-cascade_acc_orig:>+10.4f}')
print(f'  {"Macro-F1":>12}   {cascade_f1_orig:>10.4f}   {cascade_f1_tuned:>10.4f}   {cascade_f1_tuned-cascade_f1_orig:>+10.4f}')

print('\n📋 PER-CLASS F1:')
print(f'  {"Class":>12}   {"Original":>10}   {"Tuned":>10}   {"Change":>10}')
print(f'  {"-"*50}')
for i, name in enumerate(LABEL_NAMES):
    f1_orig  = f1_score(y_test_g == i, final_preds_orig  == i, zero_division=0)
    f1_tuned = f1_score(y_test_g == i, final_preds_tuned == i, zero_division=0)
    flag = ' ✅' if f1_tuned > f1_orig else (' ❌' if f1_tuned < f1_orig else '')
    print(f'  {name:>12}   {f1_orig:>10.4f}   {f1_tuned:>10.4f}   {f1_tuned-f1_orig:>+10.4f}{flag}')

print('\n🔀 STAGE 1 ROUTING:')
orig_s1   = (s1_abnorm_prob >= 0.55).astype(int)
true_abn  = y_test_g != 0
orig_routed  = ((orig_s1 == 1) & true_abn).sum()
tuned_routed = ((stage1_preds_final == 1) & true_abn).sum()
print(f'  {"":>20}   {"Original":>12}   {"Tuned":>10}')
print(f'  {"→ Stage 2":>20}   {orig_s1.sum():>12}   {stage1_preds_final.sum():>10}')
print(f'  {"Correctly routed":>20}   {orig_routed:>12}   {tuned_routed:>10}')
print(f'  {"Routing recall":>20}   {orig_routed/true_abn.sum():>12.4f}   {tuned_routed/true_abn.sum():>10.4f}')

print('\n🔬 ABLATION CONTRIBUTIONS:')
print(f'  Ablation 1 (Threshold) : tested [0.25–0.75]  → best: {BEST_THRESHOLD}')
print(f'  Ablation 2 (cap_ratio) : tested [1.5–8.0]    → best: {BEST_CAP_RATIO}')
print(f'  Ablation 3 (gamma)     : tested [0.5–3.0]    → best: {BEST_GAMMA}')
print(f'  Ablation 4 (LR)        : tested [1e-4–3e-3]  → best: {BEST_LR}')

# Before vs After confusion matrices
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, preds, title in zip(
    axes,
    [final_preds_orig, final_preds_tuned],
    [f'Original Cascade (thresh={0.55}, cap=2.5, γ=2.0)',
     f'Tuned Cascade (thresh={BEST_THRESHOLD}, cap={BEST_CAP_RATIO}, γ={BEST_GAMMA})']
):
    cm = confusion_matrix(y_test_g, preds)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=LABEL_NAMES, yticklabels=LABEL_NAMES)
    ax.set_title(title, fontsize=10)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('True')
    ax.tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'ablation_before_after.png', dpi=150)
plt.show()
print('✓ Saved ablation_before_after.png')


In [ ]:
# ── Final summary table ───────────────────────────────────────────────────────
acc_avg_f, f1_avg_f, _ = evaluate(global_model_avg,  X_test_g, y_test_g)
acc_10_f,  f1_10_f,  _ = evaluate(global_model_10,   X_test_g, y_test_g)

summary = {
    'model'           : '2D CNN (RespCNN)',
    'features'        : 'STFT+LogMel (2, 64, 64)',
    'num_clients'     : NUM_CLIENTS,
    'num_rounds'      : NUM_ROUNDS,
    'best_fedprox_mu' : best_mu,
    'ablation_best'   : {
        'threshold': BEST_THRESHOLD,
        'cap_ratio': BEST_CAP_RATIO,
        'gamma'    : BEST_GAMMA,
        'lr'       : BEST_LR,
    },
    'results': {
        'Centralized'     : {'acc': round(acc_c, 4),         'f1': round(f1_c, 4)},
        'FedAvg_126'      : {'acc': round(acc_avg_f, 4),     'f1': round(f1_avg_f, 4)},
        'FedAvg_10'       : {'acc': round(acc_10_f, 4),      'f1': round(f1_10_f, 4)},
        'Cascade_original': {'acc': round(cascade_acc_orig, 4), 'f1': round(cascade_f1_orig, 4)},
        'Cascade_tuned'   : {'acc': round(cascade_acc_tuned, 4), 'f1': round(cascade_f1_tuned, 4)},
        'FedProx': {
            str(mu): {'acc': round(prox_results[mu]['acc'], 4),
                      'f1' : round(prox_results[mu]['f1'],  4)}
            for mu in MU_SWEEP
        }
    }
}
with open(RESULTS_DIR / 'results_summary_v4.json', 'w') as f:
    json.dump(summary, f, indent=2)

print('=' * 65)
print('          FINAL RESULTS — v4 (2D CNN)')
print('=' * 65)
print(f'  Model          : 2D CNN (STFT+LogMel 2-channel 64×64)')
print(f'  Clients        : {NUM_CLIENTS}  (+10-client grouping)')
print(f'  Rounds         : {NUM_ROUNDS} × {LOCAL_EPOCHS} local epochs')
print(f'  Best FedProx μ : {best_mu}')
print('-' * 65)
print(f'  Centralized CNN   → Acc: {acc_c:.4f}   F1: {f1_c:.4f}')
print(f'  FedAvg (126)      → Acc: {acc_avg_f:.4f}   F1: {f1_avg_f:.4f}')
print(f'  FedAvg (10)       → Acc: {acc_10_f:.4f}   F1: {f1_10_f:.4f}')
for mu in MU_SWEEP:
    r    = prox_results[mu]
    mark = ' ← best' if mu == best_mu else ''
    print(f'  FedProx μ={mu:<5}    → Acc: {r["acc"]:.4f}   F1: {r["f1"]:.4f}{mark}')
print(f'  Cascade (original)→ Acc: {cascade_acc_orig:.4f}   F1: {cascade_f1_orig:.4f}')
print(f'  Cascade (tuned)   → Acc: {cascade_acc_tuned:.4f}   F1: {cascade_f1_tuned:.4f}  ← BEST')
print('=' * 65)
print(f'\n✅ All results saved to {RESULTS_DIR.resolve()}')
